### This ARTICLES notebook 

- Loads journal data into the database  
- Finds the ISSN of Domingo's Incites journals  
- Examines the "completeness" of the Incites journals

- Extracts the works for each journal into the cache

- Load the works into the database after  
    - Filters works into a flat table (work_id, doi, source, host, citation_count etc)  
    - Flattens the authorships table for each work (author_id, institution_id etc)  
    - Filters the reference list to make the cited table (When inverted these are the endogenous citations)  

Note that at this stage the works have been filtered by publicaiotn date and type (articles, etc)


In [1]:
%run common_setup.ipynb

┌──────────┬─────────┬──────────────────────┬──────────────────────┬───────────────────────────────────────┬───────────┐
│ database │ schema  │         name         │     column_names     │             column_types              │ temporary │
│ varchar  │ varchar │       varchar        │      varchar[]       │               varchar[]               │  boolean  │
├──────────┼─────────┼──────────────────────┼──────────────────────┼───────────────────────────────────────┼───────────┤
│ authors  │ main    │ authors              │ [id, orcid, displa…  │ [VARCHAR, VARCHAR, VARCHAR, VARCHAR…  │ false     │
│ backup   │ main    │ authors              │ [author_id, orcid,…  │ [VARCHAR, VARCHAR, VARCHAR, VARCHAR…  │ false     │
│ backup   │ main    │ authors_full         │ [author_id, orcid,…  │ [VARCHAR, VARCHAR, VARCHAR, VARCHAR…  │ false     │
│ backup   │ main    │ authorships          │ [work_id, author_i…  │ [VARCHAR, VARCHAR, VARCHAR, VARCHAR]  │ false     │
│ backup   │ main    │ citer_cit

In [2]:
class ArticlesETL(SetUp):

    def __init__(self):
        super().__init__()
        return
    
    def extract_journals(self):
        # Extract inCites-OpenAlex journal table (ISSN and OA journal_id)
        try:
            self.journals = self.db.sql("SELECT * FROM project.sources_oa_incites").df().\
                rename(columns={'id': 'source_id'}).sort_values('source_id')[:].reset_index(drop=True)
            print(f'{self.journals.shape = }\n{self.journals.head()}')
        except Exception as e:
            print('need to load sources_oa_incites from CSV')
            self.db.sql("CREATE OR REPLACE TABLE project.sources_oa_incites AS (SELECT * FROM read_csv('../DATA/sources_oa_incites.csv'))")
            self.journals = self.db.sql("SELECT * FROM project.sources_oa_incites").df().\
                rename(columns={'id': 'source_id'}).sort_values('source_id')[:].reset_index(drop=True)
            print(f'{self.journals.shape = }\n{self.journals.head()}')
        return

    def extract_works_by_journal(self):
        # Extract OA works for the journal set, for publication years 2010+ to now
        hold = []
        for row in self.journals.itertuples():
            # if row.Index not in [0, 11, 12]:
            #     continue
            source_id = row.source_id
            reader = rf'Works().filter(primary_location={{"source": {{"id": "{source_id}"}}}}).filter(publication_year=">2009")'
            if isinstance(oa := self._cache_manager(task=reader), pd.DataFrame) and len(oa) > 0:
                oa = self._filter_works(works=oa)
                self._sql_appender(df=oa, row=row.Index)
                print(f'APPENDED {row.Index = } {oa.shape = }')
            else:
                print(f'OpenAlex does not have articles for {source_id = } {row.display_name = }')
            if row.Index % 25 == 0:
                print(f'{row.Index}/{len(self.journals)} completed')             
        return
    
    def _filter_works(self, works=None):
        keep_columns = ["id", "doi", "title", "publication_year", "primary_location", "type",
                "countries_distinct_count", "institutions_distinct_count", "fwci", "has_fulltext", 
                "cited_by_count", "biblio", "is_retracted", "is_paratext", 
                "referenced_works_count", "cited_by_api_url", "updated_date", "created_date", 
                "authorships", "referenced_works", "topics"]
        cols = [c for c in works.columns if c in keep_columns] + [c for c in works.columns if 'biblio.' in c or 'primary_location.source' in c or 'primary_topic.' in c]
        works = works[cols] #.fillna(' ')
        condition1 = works['is_paratext'] == False
        condition2 = works['is_retracted'] == False
        condition3 = works['referenced_works_count'] != 0
        condition4 = np.array([t in {'article', 'review', 'letter'} for t in works['type']])
        condition5 = len(works['authorships']) > 0
        works = works.loc[condition1 & condition2 & condition3 & condition4 & condition5].copy()
        return works[works.id != 'https://openalex.org/works/W4285719527'] # THIS IS A DISCARDED WORK WITH ENORMOUS CITATIONS
    
    def _sql_appender(self, df=None, row=None):
        self._drop_older_duplicated_rows(df=df)
        try:
            if row == 0:
                sql = "CREATE OR REPLACE TABLE project.raw AS (SELECT * FROM df)"
            else:
                sql = "INSERT INTO project.raw BY NAME (SELECT * FROM df)"
            self.db.sql(sql)
        except Exception as e:
            print(f'CONVERTING df to SQL table project.raw {row = } {e = }')           
            print(f'{df.shape = }\n{df.columns = }\n{df.head()}')
        return
    
    def _drop_older_duplicated_rows(self, df=None):
        rows = df.shape[0]
        df = df.sort_values("updated_date", ascending=False).drop_duplicates(subset='id')
        if rows != df.shape[0]:
            print(f'DROP OLDER ROWS {df.shape = } due to duplicated update_date')
        return
 
    def duplicate_db_as_backup(self):
        self.db.sql("BEGIN TRANSACTION; COPY FROM DATABASE project TO backup; COMMIT;")
        return

In [3]:
class ExtractAuthorshipsReferencesTopics(SetUp):

    def __init__(self):
        super().__init__()
        return
    
    def authorships_etl(self):
        print("authorships")
        sql = """
            CREATE OR REPLACE TABLE project.authorships AS  
                (SELECT work_id,
                        author_id,
                        unnest(authorship.institutions).id AS institution_id,
                        unnest(authorship.institutions).country_code AS country_code,    
                    FROM
                    (SELECT id AS work_id,
                            unnest(authorships).author.id AS author_id,
                            unnest(authorships) AS authorship,
                        FROM project.raw
                    )
                )
            """
        self.db.sql(sql)
        self.db.sql("SELECT count(DISTINCT work_id), count(DISTINCT author_id), count(DISTINCT institution_id) FROM project.authorships").show()
        return
    
    def references_etl(self):
        print("references")
        sql = """
            CREATE OR REPLACE TABLE project.citer_cited AS
            WITH
            citer_cited_CTE AS
                (SELECT id AS citer_id,
                        publication_year AS citer_year,
                        unnest(referenced_works) AS cited_id
                    FROM project.raw
                ),
            citer_cited_filtered_CTE AS
                (SELECT DISTINCT citer_id,
                        citer_year,
                        cited_id,
                        publication_year AS cited_year
                    FROM citer_cited_CTE
                    RIGHT JOIN project.raw
                    ON cited_id = id
                    WHERE cited_id NOT NULL
                )

            SELECT *,
                    cited_year - citer_year - 1 AS delta_t
                FROM citer_cited_filtered_CTE
                WHERE delta_t <= 0
            """
        self.db.sql(sql)
        self.db.sql("SELECT count(DISTINCT citer_id), count(DISTINCT cited_id) FROM project.citer_cited").show()
        return
    
    def topics_etl(self):
        print("topics")
        sql = """
            CREATE OR REPLACE TABLE project.topics AS
                SELECT id AS work_id,
                        "primary_topic.id" AS topic_id,
                        "primary_topic.display_name" AS topic_name,
                        "primary_topic.score" AS topic_score,
                        "primary_topic.domain".id AS domain_id,
                        "primary_topic.field".id AS field_id,
                        "primary_topic.subfield".id AS subfield_id
                FROM project.raw
            """
        self.db.sql(sql)
        self.db.sql("SELECT count(*) FROM project.topics").show()
        return

In [4]:
class  ExtractAuthors(SetUp):

    def __init__(self):
        super().__init__()
        return

    def _extract_author_ids(self):
        df = self.db.sql("SELECT DISTINCT author_id FROM project.authorships").df()
        self.author_ids = [i.replace('https://openalex.org/', '') for i in sorted(df.author_id)] #[:256]
        return

    def extract_authors(self):
        # Extract OA authors for the journal set
        self._extract_author_ids()
        hold = []
        block_length = 100
        start = 0
        block_total = len(self.author_ids)//block_length + 1
        print(f'extract authors {start = } {block_length = } {block_total = }')
        for block_count in range(block_total):
            authors = '|'.join(self.author_ids[start: start+block_length])
            start = start + block_length
            reader = rf'Authors().filter(id="{authors}")'
            if isinstance(oa := self._cache_manager(task=reader), pd.DataFrame) and len(oa) > 0:
                # print(f'EXTRACTED {len(oa) = } authors FOR {authors = }')
                # print(f'{oa.shape = }\n{oa.head()}')
                hold.append(oa)
            else:
                print(f'OpenAlex does not have authors for {block_count=  } {block_count*block_length = } {authors = } {oa.shape = }')
            if block_count % 50 == 0:
                print(f'{block_count = } {block_count*block_length = } {block_count*block_length}/{len(self.author_ids)} completed')
        self._load_authors(hold=hold)
        return
    
    def _load_authors(self, hold=None):
        df = pd.concat(hold, axis=0).rename(columns={'id': 'author_id', 'display_name': 'author_name'})
        df.columns = [c.replace('summary_stats.', '') for c in df.columns]
        self.db.sql("CREATE OR REPLACE TABLE project.authors AS SELECT * FROM df")
        cols = ['author_id', 'orcid', 'author_name', 'display_name_alternatives', 'works_count', 'cited_by_count', '2yr_mean_citedness', 'h_index', 'i10_index', 'topics']
        df = df[cols]
        df[['first', 'middle', 'last', 'fullname']] = [normalise_name(n) for n in df.author_name]
        print(f'{df.shape = }\n{df.head()}')
        self.db.sql("CREATE OR REPLACE TABLE project.authors_full AS SELECT * FROM df")
        self.db.sql("SELECT * FROM project.authors").show()
        self.db.sql("SELECT * FROM project.authors_full").show()
        self.db.sql("SELECT count(*) FROM project.authors_full").show()
        return
    
    def duplicate_db_as_backup(self):
        self.db.sql("BEGIN TRANSACTION; COPY FROM DATABASE project TO backup; COMMIT;")
        return         

#### This cell matches Domingo's C and T lists to authors in the OpenAlex extract from the Journal Set  

- Extract Domingo's list and ensure that the names are normalised

- Compare with OpenAlex lists  

    - HCRs - endogenous - from OpenAlex references in journal set  
    - Authorships - endogenous - from OpenAlex works in journal set   
    - Authors - exogenous - from the entire OpenAlex author dataest, filtered into eeconomics and Business topics      

In [5]:
class MatchDomingoSample(SetUp):

    def __init__(self):
        super().__init__()
        return    

    def extract_sample(self):
        sample = pd.read_excel('../DATA/researchers_results_total_average_influence.xlsx').iloc[:, :10]
        sample[['first', 'middle', 'last', 'fullname']] = [normalise_name(n) for n in sample.Research_Profile]
        print(f'{sample.shape = }\n{sample.head()}')
        sample = sample.sort_values('HCP', ascending=False).reset_index(drop=True)
        print(sample[sample.duplicated(keep=False)].head(32))
        self.db.sql("CREATE OR REPLACE TABLE project.domingo_sample_original AS SELECT * FROM sample")
        self.sample = self.db.sql("SELECT * FROM project.domingo_sample_original").df()
        print(f'{sample.shape = }\n{sample.head()}')
        return
    
    def match_sample(self):
        sql = """   
            -- MATCH Domingo's EconBus list to OpenAlex
            -- matches on full name and first intial/last name
            -- finds 1280 matches (many duplicates)
            -- finds 293 distinct matches from endogenous authors
            -- finds 26 non-matching WOS names, no duplicates.
            -- there are 15 "complex" names matched by hand
            -- 
            -- ========================================
            CREATE OR REPLACE TABLE project.candidates AS
            WITH 
                match_endogenous_CTE AS
                (SELECT a.author_id,
                        a.orcid,
                        a.works_count,
                        PUB,
                        PUB/a.works_count AS works_fraction,
                        a.cited_by_count,
                        CIT,
                        CIT/a.cited_by_count AS cite_fraction,
                        a.author_name,
                        a.fullname AS oa_fullname,
                        aa.topics[1].field.display_name AS field,
                        Research_Profile,
                        "Group",
                        "class",
                        d."first",
                        d.middle,
                        d."last",
                        d.fullname,
                    FROM project.domingo_sample_original d
                    LEFT JOIN project.authors_full a
                    ON list_contains(display_name_alternatives, d.fullname) OR list_contains(display_name_alternatives, concat(d.first[1],'. ', d.last))
                    LEFT JOIN project.authors aa
                    USING (author_id)
                    ORDER BY d.fullname, a.works_count DESC
                    ),
                filtered_endogenous_CTE AS
                    (SELECT m.*
                    FROM match_endogenous_CTE m
                    WHERE list_contains(['Agricultural and Biological Sciences', 'Arts and Humanities', 'Environmental Science',
                                            'Energy', 'Psychology', -- 'Decision Sciences', 'Mathematics'
                                            'Biochemistry, Genetics and Molecular Biology', 'Computer Science',
                                            'Chemical Engineering', 'Earth and Planetary Sciences', 'Engineering', 
                                            'Materials Science' , 'Medicine', 'Neuroscience', 'Physics and Astronomy'], field) = false
                    ),
                endogenous_matched_CTE AS
                    (SELECT DISTINCT ON (Research_Profile)
                            * 
                    FROM filtered_endogenous_CTE
                    ORDER BY Research_Profile, works_count DESC
                    ),
                exogenous_match_CTE AS  
                    (SELECT oa.id AS author_id,
                            oa.orcid,
                            oa.works_count,
                            PUB,
                            PUB/oa.works_count AS works_fraction,
                            oa.cited_by_count,
                            CIT,
                            CIT/oa.cited_by_count AS cite_fraction,
                            oa.display_name AS author_name,
                            oa.display_name AS oa_fullname,
                            oa.topics[1].field.display_name AS field,
                            Research_Profile,
                            "Group",
                            "class",
                            "first",
                            middle,
                            "last",
                            fullname
                        FROM match_endogenous_CTE e
                        LEFT JOIN authors.authors oa
                        ON list_contains(oa.display_name_alternatives, e.fullname) OR list_contains(oa.display_name_alternatives, concat(e.first[1],'. ', e.last))
                        WHERE (e.author_id IS NULL) AND 
                            (field is NULL OR list_contains(['Physics and Astronomy', 'Earth and Planetary Sciences', 
                                            'Biochemistry, Genetics and Molecular Biology', 'Medicine', 
                                            'Veterinary', 'Engineering', 'Neuroscience', 'Nursing',
                                            'Health Professions', 'Materials Science'], field) = false) 
                        ORDER BY Research_Profile, oa.works_count DESC
                    ),
                difficult_names_CTE AS
                (SELECT oa.id AS author_id,
                        oa.orcid,
                        oa.works_count,
                        PUB,
                        PUB/oa.works_count AS works_fraction,
                        oa.cited_by_count,
                        CIT,
                        CIT/oa.cited_by_count AS cite_fraction,
                        oa.display_name AS author_name,
                        oa.display_name AS oa_fullname,
                        oa.topics[1].field.display_name AS field,
                        Research_Profile,
                        "Group",
                        "class",
                        "first",
                        middle,
                        "last",
                        fullname
                    FROM '/home/lc/Projects/EconomicsBusiness/DATA/difficult_name_matches.csv'
                    LEFT JOIN project.domingo_sample_original d
                    USING (Research_Profile)
                    LEFT JOIN authors.authors oa
                    ON id = author_id
                ),
                openalex_kind_CTE AS
                (SELECT author_id,
                        orcid,
                        works_count,
                        NULL AS PUB,
                        NULL AS works_fraction,
                        cited_by_count,
                        NULL AS CIT,
                        NULL AS cite_fraction,
                        author_name,
                        fullname AS oa_fullname,
                        topics[1].field.display_name AS field,
                        NULL AS Research_Profile,
                        'X' AS "Group",
                        NULL AS "class",
                        NULL AS "first",
                        NULL AS middle,
                        NULL AS "last",
                        NULL AS fullname
                        -- row_number() OVER (ORDER BY works_count DESC) AS row_count
                    FROM project.authors_full a
                    WHERE list_contains((SELECT list(author_id) FROM project.candidates GROUP BY ALL), author_id) = false
                            AND list_contains(['Economics, Econometrics and Finance'], field) --'Decision Sciences',  'Social Sciences'], field)
                    LIMIT 150
                    )

                        -- SELECT 'openalex' AS kind,
                        --         *
                        --   FROM openalex_kind_CTE

            SELECT *
            FROM
                (SELECT 'endogenous' AS kind,
                            *
                        FROM endogenous_matched_CTE
                    UNION  
                        SELECT DISTINCT ON (Research_Profile)
                            'exogenous' AS kind,
                                *
                        FROM exogenous_match_CTE
                        WHERE author_id IS NOT NULL
                    UNION
                        SELECT DISTINCT ON (Research_Profile)
                                'unmatched' AS kind,
                                *
                        FROM exogenous_match_CTE
                        WHERE author_id IS NULL
                    UNION
                        SELECT DISTINCT ON (Research_Profile)
                                'matched_difficult' AS kind,
                                *
                        FROM difficult_names_CTE
                    UNION
                        SELECT 'openalex' AS kind,
                                *
                        FROM openalex_kind_CTE
                    )
            """
        self.db.sql(sql)  #.show()
        return

    def load_sample(self):
        print('load matched sample together with author data from OA')
        sample= self.db.sql("SELECT * FROM project.candidates").df().sort_values('Research_Profile').reset_index(drop=True)
        print(f'{sample.shape = } {sample['kind'].unique() = }\n{sample.head()}\n{sample.loc[[s is None for s in sample.kind], :].head()}')
        with pd.ExcelWriter('../DATA/domingo_sample_match.xlsx') as writer:
            sample.to_excel(writer, index=False, sheet_name='all_data')
            sample.query("kind == 'endogenous'").to_excel(writer, index=False, sheet_name='endogenous')
            sample.query("kind == 'exogenous'").to_excel(writer, index=False, sheet_name='exogneous')
            sample.query("kind == 'unmatched'").to_excel(writer, index=False, sheet_name='unmatched')
            sample.query("kind == 'matched_difficult'").to_excel(writer, index=False, sheet_name='unmatched')
            sample.query("kind == 'openalex'").to_excel(writer, index=False, sheet_name='openalex')
            sample.loc[[k is None for k in sample["kind"]], :].to_excel(writer, index=False, sheet_name='unmatched_')
            sample.loc[[k is not None for k in sample["author_id"]], :].to_excel(writer, index=False, sheet_name='best_case')
        return

In [6]:
def main():

    # jetl = ArticlesETL()
    # jetl.extract_journals()
    # jetl.extract_works_by_journal()
    # print(jetl.db.sql("DESCRIBE TABLE project.raw").df())
    # jetl.db.sql("SELECT * FROM project.raw").show()
    # sql = """SELECT count(DISTINCT "primary_location.source"['id']) FROM project.raw"""
    # jetl.db.sql(sql).show()
    # jetl.duplicate_db_as_backup()
    # jetl.db.close()

    # ea = ExtractAuthorshipsReferencesTopics()
    # ea.authorships_etl()
    # ea.references_etl()
    # ea.topics_etl()
    # ea.db.close()

    # eauthors = ExtractAuthors()
    # eauthors.extract_authors()
    # eauthors.duplicate_db_as_backup()
    # eauthors.db.close()
    
    mds = MatchDomingoSample()
    mds.extract_sample()
    mds.match_sample()
    mds.load_sample()
    mds.db.close()

In [7]:
if __name__ == "__main__":
    main()
    print("DONE!")

┌──────────┬─────────┬──────────────────────┬──────────────────────┬───────────────────────────────────────┬───────────┐
│ database │ schema  │         name         │     column_names     │             column_types              │ temporary │
│ varchar  │ varchar │       varchar        │      varchar[]       │               varchar[]               │  boolean  │
├──────────┼─────────┼──────────────────────┼──────────────────────┼───────────────────────────────────────┼───────────┤
│ authors  │ main    │ authors              │ [id, orcid, displa…  │ [VARCHAR, VARCHAR, VARCHAR, VARCHAR…  │ false     │
│ backup   │ main    │ authors              │ [author_id, orcid,…  │ [VARCHAR, VARCHAR, VARCHAR, VARCHAR…  │ false     │
│ backup   │ main    │ authors_full         │ [author_id, orcid,…  │ [VARCHAR, VARCHAR, VARCHAR, VARCHAR…  │ false     │
│ backup   │ main    │ authorships          │ [work_id, author_i…  │ [VARCHAR, VARCHAR, VARCHAR, VARCHAR]  │ false     │
│ backup   │ main    │ citer_cit